# 3.4 Grounded generation and citations

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Learning objectives

- Build the full RAG chain in LCEL with no legacy chain classes, and say where `create_retrieval_chain` lives now.
- Write a grounding prompt with an abstention instruction, and explain why the instruction is load-bearing.
- Add numbered citations to answers and verify them against the retrieved text.
- Extract structured claim/quote/source triples with `with_structured_output` to catch fabricated citations.
- Read provider-verified citations (OpenAI annotations, Anthropic citations) and name their limits.
- Budget tokens, cost, and latency per query before the first production call.

## The full RAG chain, in LCEL

Retrieval (lesson 3.3) is half the system. The other half is generation grounded in what was retrieved. In the 1.x era there is exactly one supported way to wire this: plain LCEL. The old `create_retrieval_chain` and `RetrievalQA` helpers moved to `langchain-classic` with the other legacy chains; new code does not use them.

The chain has four steps, and you should be able to point at each one in the code: retrieve documents, format them into one text block, fill a prompt template, call the model.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# retriever: the hybrid retriever from lesson 3.3 (or plain similarity to start)

def format_docs(docs):
    return "\n\n".join(f"[{i + 1}] {d.page_content}" for i, d in enumerate(docs))

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only the context below. "
     "If the answer is not in the context, say you do not know. "
     "Do not use any other knowledge.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("How long does shipping take in the EU?"))

Read the wiring: `retriever | format_docs` turns the question string into one formatted text block; `RunnablePassthrough()` forwards the question unchanged; the dict feeds both into the prompt as `{context}` and `{question}`. This is the same Runnable composition you learned in Module 1, applied to retrieval. Nothing about it is RAG-specific except the pieces you plug in, which is the point.

The system prompt carries three instructions, and each one earns its place. "Using only the context" bounds the knowledge source. "Say you do not know" is the abstention rule, and it is load-bearing: without it, the model fills gaps from its training data and presents guesses with the same confidence as facts. Paired with the score-threshold retriever from lesson 3.3, which returns nothing when nothing is relevant, this turns "no good context" into an honest "I don't know" instead of a confident fabrication.

## Numbered citations

A grounded answer should show its work. The simplest pattern that survives contact with real users: number the chunks when you format them (already done above with `[1]`, `[2]`), and require the model to cite the number after each claim.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only the context below. "
     "If the answer is not in the context, say you do not know. "
     "After each factual claim, cite the chunk it comes from like this: [2]. "
     "Every factual claim needs a citation.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

Numbered citations are a convention you enforce, not a guarantee the model keeps. The model can still cite `[2]` for a claim that chunk 2 does not support. Prompt-level citations are unverified claims about provenance, so for anything that matters you verify them in code. That is the next section.

## Verifying citations with structured output

Trust, but verify, mechanically. Ask the model for the answer as structured data: each claim, the exact quote supporting it, and the chunk number. Then check in code that every quote appears verbatim in its cited chunk. A fabricated citation fails this check because the quote will not be found.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.documents import Document

class Claim(BaseModel):
    claim: str = Field(description="One factual claim from the answer.")
    quote: str = Field(description="The exact supporting quote, copied word for word.")
    chunk: int = Field(description="The chunk number the quote comes from, e.g. 2.")

class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]

def format_docs_with_ids(docs: list[Document]):
    return "\n\n".join(f"[{i + 1}] {d.page_content}" for i, d in enumerate(docs))

cited_chain = (
    {"context": retriever | format_docs_with_ids, "question": RunnablePassthrough()}
    | ChatPromptTemplate.from_messages([
        ("system",
         "Answer using only the context. For every factual claim in your answer, "
         "record the claim, the exact supporting quote, and the chunk number. "
         "If the answer is not in the context, say you do not know and leave claims empty.\n\n"
         "Context:\n{context}"),
        ("human", "{question}"),
    ])
    | llm.with_structured_output(CitedAnswer)
)

def verify_citations(result: CitedAnswer, docs: list[Document]) -> list[str]:
    """Return a list of problems. Empty means every citation checks out."""
    problems = []
    for c in result.claims:
        if not 1 <= c.chunk <= len(docs):
            problems.append(f"Claim cites chunk {c.chunk}, which does not exist.")
            continue
        if c.quote not in docs[c.chunk - 1].page_content:
            problems.append(f"Quote for '{c.claim[:40]}...' not found verbatim in chunk {c.chunk}.")
    return problems

result = cited_chain.invoke("How long does shipping take in the EU?")
problems = verify_citations(result, retriever.invoke("How long does shipping take in the EU?"))

This is one extra model call and a string check, and it converts citations from decoration into evidence. Two engineering notes. First, retrieve the documents once and reuse them; the snippet above invokes the retriever twice for clarity, but in real code you would cache the list. Second, this pattern doubles generation cost (one call for the answer, structured output included in the same call here, so actually the cost is one call with a larger output schema; the verification itself is free string matching). The honest cost accounting: structured output makes each generation call somewhat more expensive in output tokens, and worth it wherever a wrong citation is worse than a slow answer.

## Provider-verified citations: what the model itself anchors

Prompt-based citations are yours to verify. Two providers now offer citations the model anchors itself, with character offsets into the source. Know both, and know their limits.

**OpenAI.** When the model uses a search tool (web search or file search) through the Responses API, the API returns `url_citation` annotations on the output text: the URL, title, and the character range the claim came from. In `langchain-openai` 1.7.2 these arrive on the message's content blocks, which you read via `message.content_blocks`. The limit is right there in the description: annotations appear only when the model actually used a search tool. They do not apply to your own RAG context stuffed into the prompt; for that, you verify citations yourself as above.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", use_responses_api=True)
# ... after a call where the model used web search:
for block in response.content_blocks:
    if block.get("type") == "text":
        for ann in block.get("annotations", []):
            if ann.get("type") == "url_citation":
                print(ann["url"], ann["title"])

**Anthropic.** The citations API lets you pass documents (or your own `search_result` blocks from a RAG tool) with `"citations": {"enabled": True}`, and Claude anchors its claims to exact spans in those documents. This is the closest thing to verified grounding either provider offers for your own documents, and LangChain's ChatAnthropic docs cover the pattern. The limit: it applies to documents you pass as Anthropic document blocks, and parsing the citation spans is your code's job; every provider's citation format is different, so normalise them at your system boundary.

The rule of thumb: provider citations are strongest where the provider did the retrieval (web search, file search). For your own retrieval pipeline, prompt citations plus code verification (the previous section) is the pattern that holds up.

## Hallucination guards: the full set

Grounded generation fails in specific ways. Here is the guard for each one.

- **Answering from training data instead of context.** Guard: the abstention instruction plus the score-threshold retriever. If retrieval returns nothing useful, the chain must say it does not know.
- **Citing a chunk that does not support the claim.** Guard: the quote-verification check above.
- **Drifting mid-answer.** Long answers wander away from the context as generation continues. Guard: keep answers short (ask for the answer first, details only on follow-up), and for high-stakes use, run a faithfulness self-check: a second model call that reads the answer and the context and reports whether every claim is supported. That doubles generation cost, so reserve it for answers where a mistake has consequences.
- **The lost middle, again.** Retrieval found the right chunk, but it sits at position 9 of 12 in the prompt and the model ignores it. Guard: keep `k` small and relevant (3 to 5 chunks beats 20 loose ones), and put the most important chunks first and last in the formatted context, since models attend best to the edges.

## Context-window math, cost, and latency

Do this arithmetic before production, not after the first invoice.

- **Context math.** Input tokens per query = system prompt + question + sum over chunks of (chunk tokens + the `[n]` label and separators). With `k=6` chunks of 400 tokens, that is roughly 2,500 to 3,000 input tokens before the question. Know your model's window (gpt-4o-mini: 128k input) and your chunk budget, and assert on it: if the formatted context exceeds your budget, truncate or reduce `k` in code rather than hoping the model copes.
- **Embedding cost.** One embedding call per query (the question), plus one per chunk at index time. `text-embedding-3-small` is priced per million tokens; a question is tens of tokens, so per-query embedding cost is effectively zero next to generation.
- **Rerank cost.** Roughly $0.001 to $0.002 per query for a hosted cross-encoder reranker (2026 benchmarks), 50 to 220 ms of latency. Self-hosted rerankers trade that for GPU cost you already pay.
- **Generation cost.** Input tokens at the input price, output tokens at the output price. Structured citation output adds output tokens. The faithfulness self-check doubles the generation part.
- **Latency budget.** A realistic per-query budget: question embedding 20 to 50 ms, retrieval 30 to 80 ms, rerank 50 to 220 ms, generation to first token 1 to 2 seconds. Reranking is the only step you can cut without changing the answer's shape; everything else is structural.

EU angle, stated plainly: every hosted call (embeddings, reranker, LLM) sends your users' questions, and possibly your documents, to a provider outside the EU. Under GDPR you need a legal basis for that transfer and usually a data processing agreement with the provider. The fully local path (self-hosted embeddings, BM25, local FAISS, self-hosted reranker, a local model via Ollama) keeps all data on your infrastructure; it costs you hardware instead of API fees. Also remember that query logs are personal data when they can identify a user: minimise what you log, set a retention period, and say so in your privacy notice. Module 6 covers the EU AI Act deployer obligations; for now, note that a customer-facing RAG bot is exactly the kind of system the Act cares about.

## What can go wrong

- **The chain answers confidently with no context.** The abstention instruction is missing or weak, and a fixed `k` always supplies something to riff on. Fix both: threshold retrieval plus "say you do not know".
- **Citations that point nowhere.** Prompt-level `[2]` citations are unverified. Verify quotes verbatim in code, or treat the citations as decoration and say so.
- **Verifying against a different retrieval run.** If you invoke the retriever separately for verification, ranking nondeterminism (or an index update between calls) can make quotes mismatch. Retrieve once, reuse the document list.
- **Structured output silently dropping claims.** If the model returns fewer claims than the answer contains facts, your verification passes on an incomplete set. Ask for "every factual claim" explicitly, and spot-check on your probe set.
- **Assuming provider citations cover your RAG.** OpenAI's `url_citation` annotations only exist when the model used a search tool. Your stuffed context gets no annotations; that verification is your code's job.
- **Context overflow on long corpora days.** A new document type with longer chunks pushes formatted context past your budget. Assert on token count in code and reduce `k` or truncate; do not rely on the model to handle overflow gracefully.
- **Cost surprise from the self-check.** A faithfulness second pass doubles generation cost per query. Gate it: run it on all answers during evaluation, and in production only where a wrong answer costs more than the check.
- **Logging queries without thinking.** Query logs with user questions are personal data under GDPR. Minimise, set retention, and keep logs in the EU if your users are there.

## Check your understanding

**1. `create_retrieval_chain` appears in a 2025 tutorial you want to adapt. What do you do, and what do you write instead?**
You do not install anything to make it work: it lives in `langchain-classic`, the maintenance package, and is legacy. Write the chain in LCEL instead: `({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())`, with a grounding system prompt and numbered chunk labels in `format_docs`.

**2. Your RAG bot answers questions about topics with no relevant chunks, confidently and wrongly. Name the two changes that fix this, one in retrieval and one in generation.**
Retrieval: switch to `similarity_score_threshold` so irrelevant queries return nothing instead of the least-bad chunks. Generation: add the abstention instruction ("if the answer is not in the context, say you do not know") to the system prompt. Together they turn "no good context" into an honest refusal.

**3. The model cites [3] after a claim. How do you check whether that citation is real, in code?**
Require the model to also return the exact supporting quote and chunk number as structured output (`with_structured_output`), then check in code that the quote appears verbatim in chunk 3's text and that chunk 3 exists. A fabricated citation fails because the quote will not be found word for word in the cited chunk.

**4. When do OpenAI's `url_citation` annotations appear on a response, and where do you read them in LangChain?**
Only when the model used a search tool (web search or file search) via the Responses API; they do not cover context you stuffed into the prompt yourself. In `langchain-openai` 1.7.2 they arrive on the AIMessage's content blocks, readable via `message.content_blocks`, as blocks of type `text` carrying an `annotations` list with entries of type `url_citation` (URL, title, character offsets).

**5. Your context budget is 4,000 input tokens for retrieval context. Chunks average 350 tokens, the system prompt and question take 300 tokens, and each chunk adds about 10 tokens of formatting. What is the largest safe k, and what do you do in code if the formatted context exceeds the budget?**
Each chunk costs about 360 tokens, so 300 + 360k must stay under 4,000, giving k = 10 at 3,900 tokens. In code, assert on the formatted context's token count (or a character-based estimate) and reduce k or truncate chunks when it exceeds the budget, rather than sending oversized prompts and hoping.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.